In [1]:
from helper_func import *
import helper_func as hf
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd
# Preprocessing
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re
import inflect
from wordsegment import load, segment
from nltk.corpus import words
word_list = set(words.words())

2024-04-13 20:19:24.571600: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-13 20:19:24.603447: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-13 20:19:25.099374: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [6]:
train = pd.read_csv('data/train.csv')

In [7]:
train.head()

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [8]:
# time the run time

import time

start = time.time()

# Clean the train and test text data

train = hf.clean_text(train, col_name = 'full_text')


end = time.time()

print(f"Time taken to clean text: {end - start} seconds")

Starting text cleaning process. 

Time taken to clean text: 67.43470859527588 seconds


In [9]:
train.head()

,essay_id,full_text,score,lowered,clean_text
0,000d118,Many people have car where they live. The thin...,3,many people have car where they live. the thin...,many people have car where they live the thing...
1,000fe60,I am a scientist at NASA that is discussing th...,3,i am a scientist at nasa that is discussing th...,i am a scientist at nasa that is discussing th...
2,001ab80,People always wish they had the same technolog...,4,people always wish they had the same technolog...,people always wish they had the same technolog...
3,001bdc0,"We all heard about Venus, the planet without a...",4,"we all heard about venus, the planet without a...",we all heard about venus the planet without al...
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3,"dear, state senator\n\nthis is a letter to arg...",dear state senator this is a letter to argue i...


In [11]:
text_col = 'clean_text'   # 'segmented_text'


glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)

Loading embeddings from /home/laptop/github/kaggle/scoring/data/glove-840B-300d.txtLoading embeddings from /home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txtLoading embeddings from /home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec




Reading Embedding File: 999995it [00:35, 28112.67it/s]]


Loaded embeddings from /home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec


Reading Embedding File: 1703756it [00:58, 29260.84it/s]


Loaded embeddings from /home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt


...

Coverage checked.
Checking coverage.


Checking Words: 100%|██████████| 65918/65918 [00:00<00:00, 1674480.24it/s]


Coverage checked.
Checking coverage.


Checking Words: 100%|██████████| 65918/65918 [00:00<00:00, 1797181.06it/s]

Coverage checked.
Processed dataset.


In [12]:
misspellings = []

oov = glove + paragram + fastetxt

for word, _ in oov:

    misspellings.append(word)

    misspellings = list(set(misspellings))

print(f"Number of misspelled words: {len(misspellings)}")

# print(f"Misspelled words: {misspellings}")

Number of misspelled words: 36335


In [13]:
# Example usage
corrected_words, uncorrected_words = main(misspellings)

  0%|          | 0/21 [00:00<?, ?it/s]

In [ ]:
# Assuming df is your DataFrame and 'clean_text' is the column you want to correct

correction_dict = dict(corrected_words)

train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))


In [ ]:
print(f"Corrected words: {len(corrected_words)}")
print(f"Uncorrected words: {len(uncorrected_words)}")

In [ ]:
# Apply the parallelization

train = parallelize_dataframe(train, apply_segmentation)

train.head()

In [ ]:
text_col = 'clean_text'   # 'segmented_text'


glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)


misspellings = []

oov = glove + paragram + fastetxt

for word, _ in oov:

    misspellings.append(word)

    misspellings = list(set(misspellings))

print(f"Number of misspelled words: {len(misspellings)}")

# print(f"Misspelled words: {misspellings}")


# Example usage
corrected_words, uncorrected_words = main(misspellings)

# Assuming df is your DataFrame and 'clean_text' is the column you want to correct

correction_dict = dict(corrected_words)

train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))

print(f"Corrected words: {len(corrected_words)}")
print(f"Uncorrected words: {len(uncorrected_words)}")


In [ ]:
train_essays, validation_essays = custom_train_validation_split(train, test_size=0.25, random_state=42)

In [ ]:
train_essays, _ = preprocess_data(train_essays)

In [ ]:
validation_essays, _ = preprocess_data(validation_essays) #, tfidf_vectorizer=tfidf_vectorizer)

In [ ]:
drop_cols = ['full_text','lowered', 'clean_text',
       'corrected_text']

train_df = train_essays.copy()
val_df = validation_essays.copy()

train_df.drop(columns=drop_cols, inplace= True)
val_df.drop(columns=drop_cols, inplace= True)



In [ ]:
feature_cols = []

for col in train_df.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 

In [ ]:
from sklearn.preprocessing import StandardScaler

train_labels = train_df['score']

val_labels = val_df['score']

train_features = train_df[feature_cols]

val_features = val_df[feature_cols]

scaler = StandardScaler()

train_feats_scaled = scaler.fit_transform(train_features)

val_feats_scaled = scaler.transform(val_features)


import pickle

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
import tensorflow as tf

# Assuming train_labels and val_labels are your label arrays with classes 1-6

train_labels_one_hot = tf.keras.utils.to_categorical(train_labels - 1, num_classes=6)
val_labels_one_hot = tf.keras.utils.to_categorical(val_labels - 1, num_classes=6)


In [ ]:
import tensorflow as tf

train_class = tf.data.Dataset.from_tensor_slices((train_feats_scaled, 
                                                train_labels_one_hot)).shuffle(len(train_features)).batch(32)

val_class = tf.data.Dataset.from_tensor_slices((val_feats_scaled, 
                                              val_labels_one_hot)).batch(32)


In [ ]:
train_labels.shape

In [ ]:
val_labels.shape

In [ ]:
import tensorflow as tf
import numpy as np

def quadratic_weighted_kappa(y_true, y_pred):
    """
    Calculates the Quadratic Weighted Kappa (QWK) aka Cohen's Kappa with quadratic weights.
    
    Parameters:
    y_true (tensor): The ground truth labels as one-hot encoded vectors.
    y_pred (tensor): The predicted probability distributions.
    
    Returns:
    float: The QWK score.
    """
    # Convert predictions to class indices
    y_pred_classes = tf.argmax(y_pred, axis=-1)
    y_pred_one_hot = tf.one_hot(y_pred_classes, depth=y_true.shape[-1])

    # Calculate the confusion matrix
    confusion_matrix = tf.math.confusion_matrix(tf.argmax(y_true, axis=1), y_pred_classes, num_classes=y_true.shape[-1])

    # Normalize the confusion matrix
    confusion_matrix = tf.cast(confusion_matrix, dtype=tf.float32)
    confusion_matrix = confusion_matrix / tf.reduce_sum(confusion_matrix)

    # Calculate weights
    num_classes = tf.shape(confusion_matrix)[0]
    weights = tf.cast(tf.range(num_classes), dtype=tf.float32) / tf.cast(num_classes - 1, dtype=tf.float32)
    weights = tf.square(tf.expand_dims(weights, -1) - tf.expand_dims(weights, 0))

    # Calculate the observed and expected agreement
    observed = tf.reduce_sum(weights * confusion_matrix)
    expected = tf.reduce_sum(
        weights * tf.matmul(
            tf.reshape(tf.reduce_sum(confusion_matrix, axis=1), [-1, 1]),
            tf.reshape(tf.reduce_sum(confusion_matrix, axis=0), [1, -1])
        )
    )

    # Calculate the kappa score
    kappa = 1.0 - observed / expected
    return kappa


In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import schedules, AdamW, RMSprop

def build_classification_model(hp, input_shape=(train_feats_scaled.shape[1],)):
    
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)  # number of hidden layers
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)  # neurons in each hidden layer
    l2_reg = hp.Float("l2_reg", min_value=1e-6, max_value=1e-2, sampling="log")  # L2 regularization
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")  # learning rate

    optimizer_choice = hp.Choice("optimizer_choice", ['adam', 'sgd', 'RMSprop', 'Adagrad', 'Adadelta', 'Nadam', 'Ftrl', 'L-BFGS'])

    # Learning rate schedulers
    lr_schedule = schedules.ExponentialDecay(initial_learning_rate=learning_rate,
                                             decay_steps=10000,
                                             decay_rate=0.9)

    if optimizer_choice == 'adam':
        optimizer = AdamW(learning_rate=lr_schedule)
    elif optimizer_choice == 'sgd':
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)
    elif optimizer_choice == 'RMSprop':
        optimizer = RMSprop(learning_rate=lr_schedule)
    elif optimizer_choice == 'Adagrad':
        optimizer = tf.keras.optimizers.Adagrad(learning_rate=lr_schedule)
    elif optimizer_choice == 'Adadelta':
        optimizer = tf.keras.optimizers.Adadelta(learning_rate=lr_schedule)
    elif optimizer_choice == 'Nadam':
        optimizer = tf.keras.optimizers.Nadam(learning_rate=lr_schedule)
    else:
        optimizer = tf.keras.optimizers.Ftrl(learning_rate=lr_schedule)

    model = tf.keras.models.Sequential()

    # Input layer
    model.add(tf.keras.Input(shape=input_shape))  # Explicit input layer

    model.add(tf.keras.layers.Dense(n_neurons, 
                                    activation='relu', 
                                    kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))

    dropout_rate = hp.Float("dropout_rate", min_value=0.0, max_value=0.5, step=0.05)  # dropout rate

    # Hidden layers with Batch Normalization and Dropout
    for _ in range(n_hidden):
        model.add(tf.keras.layers.BatchNormalization())
        
        model.add(tf.keras.layers.Dense(n_neurons, 
                                        activation='relu', 
                                        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
        
        model.add(tf.keras.layers.Dropout(rate=dropout_rate))

    # Output layer for 6 classes
    model.add(tf.keras.layers.Dense(6, activation='softmax'))
    
    
    
#     from tensorflow.keras.metrics import AUC

#     # For binary classification
#     roc_auc = AUC(curve='ROC')

#     # For multi-class classification (assuming one-vs-all approach)
#     roc_auc_multi = AUC(curve='ROC', multi_label=True, num_labels=6)




    # Then, you can add it to your model's compile method as a metric
    model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              metrics=[ 
                       tf.keras.metrics.CategoricalAccuracy(),
                       tf.keras.metrics.F1Score( average='weighted'),
                       tf.keras.metrics.AUC(curve='ROC', multi_label=True, num_labels=6),
                       quadratic_weighted_kappa])


    # Learning rate reduction callback
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)

    return model


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import keras_tuner as kt

model_checkpoint = ModelCheckpoint('best_standard_model_epoch.keras', 
                                   save_best_only=True, monitor='val_loss', mode='min')

early_stopping = EarlyStopping(monitor='val_loss', patience=5, 
                               restore_best_weights=True)

call_backs = [model_checkpoint, early_stopping]

tuner = kt.BayesianOptimization(
    build_classification_model,
    objective='val_loss',
    max_trials=8,
    num_initial_points= 2,
    seed=1,
    overwrite=True,
    directory='tuner_2',
    project_name='standard',
)

tuner.search(train_class, epochs=1000, 
             validation_data= val_class, callbacks = call_backs, verbose=2)

best_standard_model = tuner.get_best_models(num_models=1)[0]


best_standard_model.save('best_standard_model.keras')

In [ ]:
# Evaluate the model on the validation dataset
# This will return the loss, accuracy, and ROC AUC if ROC AUC was included during model compilation
results = best_standard_model.evaluate(val_class)

# Print all results
# The exact indices for loss, accuracy, and ROC AUC depend on the order of metrics you specified
# Commonly, 0 is loss, 1 is accuracy, and 2 would be ROC AUC if it's the third metric
print(f"Validation Loss: {results[0]}")
print(f"Validation Accuracy: {results[1]}")
if len(results) > 2:
    print(f"Validation ROC AUC: {results[2]}")
else:
    print("ROC AUC was not included in the model's metrics during compilation.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Assuming val_reg is your validation dataset
# Extract true labels and convert from one-hot encoding to integer labels if necessary
y_true = np.concatenate([y for x, y in val_class], axis=0)
if y_true.ndim > 1 and y_true.shape[1] > 1:  # Check if y_true is one-hot encoded
    y_true = np.argmax(y_true, axis=1)

# Predict the classes with the best model on the validation data
y_pred = best_standard_model.predict(val_class)
y_pred_classes = np.argmax(y_pred, axis=1)  # Convert from probabilities to class labels

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Plotting the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()


In [ ]:
y_pred_classes

In [ ]:
y_true

# As a regression Task

In [ ]:
from sklearn.preprocessing import StandardScaler
import pickle
import numpy as np

# Initialize the scaler
target_scaler = StandardScaler()

# Reshape the 1D arrays to 2D arrays with one column each
train_labels_reshaped = np.array(train_labels).reshape(-1, 1)
val_labels_reshaped = np.array(val_labels).reshape(-1, 1)

# Scale the training and validation target variables
train_target_scaled = target_scaler.fit_transform(train_labels_reshaped)
val_target_scaled = target_scaler.transform(val_labels_reshaped)

# Save the scaler object for future use
with open('target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)


In [ ]:
scaler_path = 'scaler.pkl'

# Check if the file has been written correctly and is not empty
import os

if os.path.getsize(scaler_path) > 0:
    print(f"Scaler saved successfully in {scaler_path}.")
else:
    print(f"Failed to save scaler to {scaler_path}. File is empty.")

In [ ]:
import tensorflow as tf

train_reg = tf.data.Dataset.from_tensor_slices((train_feats_scaled, 
                                                train_target_scaled)).shuffle(len(train_feats_scaled)).batch(32)

val_reg = tf.data.Dataset.from_tensor_slices((val_feats_scaled, 
                                              val_target_scaled)).batch(32)


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import schedules, AdamW, RMSprop

def build_regression_model(hp, input_shape=(train_feats_scaled.shape[1],)):
    
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)  # number of hidden layers
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)  # neurons in each hidden layer
    l2_reg = hp.Float("l2_reg", min_value=1e-6, max_value=1e-2, sampling="log")  # L2 regularization
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")  # learning rate
    
    optimizer_choice = hp.Choice("optimizer_choice", ['adam', 'sgd', 'RMSprop', 'Adagrad', 'Adadelta', 'Nadam', 'Ftrl', 'L-BFGS'])
    
    # Learning rate schedulers
    lr_schedule = schedules.ExponentialDecay(initial_learning_rate=learning_rate,
                                             decay_steps=10000,
                                             decay_rate=0.9)
    
    if optimizer_choice == 'adam':
        optimizer = AdamW(learning_rate=lr_schedule)
    elif optimizer_choice == 'sgd':
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)
    elif optimizer_choice == 'RMSprop':
        optimizer = RMSprop(learning_rate=lr_schedule)
    elif optimizer_choice == 'Adagrad':
        optimizer = tf.keras.optimizers.Adagrad(learning_rate=lr_schedule)
    elif optimizer_choice == 'Adadelta':
        optimizer = tf.keras.optimizers.Adadelta(learning_rate=lr_schedule)
    elif optimizer_choice == 'Nadam':
        optimizer = tf.keras.optimizers.Nadam(learning_rate=lr_schedule)
    else:
        # Ftrl will be the default optimizer
        optimizer = tf.keras.optimizers.Ftrl(learning_rate=lr_schedule)

    
    model = tf.keras.models.Sequential()
    
    model.add(tf.keras.Input(shape=input_shape))  # Explicit input layer

    model.add(tf.keras.layers.Dense(hp.Int('input_units', min_value=32, max_value=512, step=32), 
                                    activation='relu'))
    
    dropout_rate = hp.Float("dropout_rate", min_value=0.0, max_value=0.5, step=0.05)  # dropout rate
    
    # Adding Batch Normalization and dropout for each hidden layer
    for _ in range(n_hidden):
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Dense(n_neurons, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
        model.add(tf.keras.layers.Dropout(rate=dropout_rate))
    
    model.add(tf.keras.layers.Dense(1))  # Output layer
    
        # Then, you can add it to your model's compile method as a metric
    model.compile(optimizer=optimizer,  # Use the dynamic optimizer
              loss=tf.keras.losses.Huber(),
              metrics=['mse', 'mae'])



    # Learning rate reduction callback
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)

    return model

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import keras_tuner as kt

model_checkpoint = ModelCheckpoint('best_regression_model_epoch.keras', 
                                   save_best_only=True, monitor='val_loss', mode='min')

early_stopping = EarlyStopping(monitor='val_loss', patience=5, 
                               restore_best_weights=True)

call_backs = [model_checkpoint, early_stopping]

# reg_tuner = kt.BayesianOptimization(
#     build_regression_model,
#     objective='val_loss',
#     max_trials=8,
#     num_initial_points= 2,
#     seed=1,
#     overwrite=True,
#     directory='tuner_reg',
#     project_name='regresion',
# )
# # hyperband model 

reg_tuner = kt.Hyperband(
    build_regression_model,
    objective='val_loss',
    max_epochs=1000,
    factor=3,
    seed=1,
    overwrite=True,
    directory='tuner_reg',
    project_name='regresion',
    
)





reg_tuner.search(train_reg, epochs=1000, 
             validation_data= val_reg, callbacks = call_backs, verbose=2)

best_regression_model = reg_tuner.get_best_models(num_models=1)[0]


best_regression_model.save('best_regression_model.keras')

In [ ]:
reg_results = best_regression_model.evaluate(val_reg)

print(f"Validation Loss: {reg_results[0]}")
print(f"Validation MSE: {reg_results[1]}")
if len(reg_results) > 2:
    print(f"Validation MAE: {reg_results[2]}")
else:
    print("MAE was not included in the model's metrics during compilation.")

In [ ]:
scaled_predictions = best_regression_model.predict(val_reg)

unscaled_predictions = target_scaler.inverse_transform(scaled_predictions)

rounded_predictions = np.round(unscaled_predictions).flatten()  # Round to nearest integer and flatten if necessary
clipped_predictions = np.clip(rounded_predictions, 1, 6)  # Clip to ensure within 1-6 range

# Now, `clipped_predictions` contains the integer score predictions


In [ ]:
clipped_predictions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming `clipped_predictions` are your model predictions
# and `actual_scores` (or `val_labels`) are the ground truth scores for the test set

# Calculate the absolute errors
errors = np.abs(clipped_predictions - val_labels)

# You can scale the errors to ensure the sizes are visible and meaningful on the plot
# Adjust the scaling factor as needed to get a clear visual representation
size = errors * 10  # Example scaling factor

# Scatter plot of actual vs. predicted scores
plt.figure(figsize=(10, 6))
plt.scatter(val_labels, clipped_predictions, s=size, alpha=0.6)  # Use size for marker size
plt.title('Actual vs. Predicted Scores')
plt.xlabel('Actual Scores')
plt.ylabel('Predicted Scores')
plt.plot([1, 6], [1, 6], 'r--')  # Diagonal line representing perfect predictions
plt.colorbar(label='Absolute Error')  # Optionally add a colorbar to indicate the size meaning
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming `clipped_predictions` are your model predictions
# and `actual_scores` are the ground truth scores for the test set

# Scatter plot of actual vs. predicted scores
plt.figure(figsize=(10, 6))
plt.scatter(val_labels, clipped_predictions, alpha=0.6)
plt.title('Actual vs. Predicted Scores')
plt.xlabel('Actual Scores')
plt.ylabel('Predicted Scores')
plt.plot([1, 6], [1, 6], 'r--')  # Diagonal line representing perfect predictions
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Assuming `clipped_predictions` contains the regression model's predictions rounded and clipped to 1-6
# And `actual_scores` contains the true scores

# Compute the confusion matrix
cm = confusion_matrix(val_labels, clipped_predictions)

# Visualize the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=range(1, 7), yticklabels=range(1, 7))
plt.title('Confusion Matrix')
plt.xlabel('Predicted Scores')
plt.ylabel('Actual Scores')
plt.show()


In [ ]:
# Example usage

actuals = val_labels -1 
predictions = clipped_predictions-1


from sklearn.metrics import cohen_kappa_score

# Calculate the Quadratic Weighted Kappa
qwk_score = cohen_kappa_score(actuals, predictions, weights='quadratic')

print("Quadratic Weighted Kappa score:", qwk_score)


In [ ]:
preds = best_standard_model.predict(val_class)
preds = np.argmax(preds, axis=1)  # Convert from probabilities to class labels

actuals = val_labels -1 
predictions = preds

# Calculate the Quadratic Weighted Kappa
qwk_score = cohen_kappa_score(actuals, predictions, weights='quadratic')

print("Quadratic Weighted Kappa score:", qwk_score)
